# ReCaLL MIA Adapted to Federated LLM Fine-Tuning

This notebook adapts the **ReCaLL** (Relative Conditional Log-Likelihood) membership inference attack (Xie et al., *ReCaLL: Membership Inference via Relative Conditional Log-Likelihoods*, EMNLP 2024, arXiv:2406.15968; code github.com/ruoyuxie/recall) to federated learning (FL) based fine-tuning of a causal language model.

**Original attack.** Build a *fixed* prefix `P = p1 (+) ... (+) pn` from `n` non-member "shots". For a target `x`, compute the unconditional log-likelihood `LL(x)` and the prefix-conditioned `LL(x | P)` under the target model, and score `ReCaLL(x) = LL(x | P) / LL(x)`. Both LLs are negative; conditioning on the non-member prefix depresses the LL *more* for members ("in-context learning as implicit fine-tuning" perturbs already-memorized examples more), so members get a higher ratio (typically > 1). The attack is **reference-model-free** and **inference-time only** -- no second model and no access to the training distribution.

**FL adaptation.** Construct paired positive and negative worlds where a target record is either included in a target client's local fine-tuning data or absent. Federated fine-tuning (FedAvg) runs first. The server-side attacker then scores the target record under the **final federated model only**, computing `LL(x)` and `LL(x | P)` with a *fixed non-member prefix* `P` distinct from any client corpus, and thresholds `ReCaLL(x) = LL(x|P)/LL(x)`. It measures TPR / TNR / Adv and persists results to Firebase Firestore.

The paper notes the multi-epoch fine-tuning regime is generally an *easier* MIA setting than the single-epoch pre-training regime it targets, so the ReCaLL signal should transfer with equal or stronger effect. The advantage over reference-model attacks is that ReCaLL needs **no reference model** -- non-member prefixes are cheap (post-cutoff or synthetic), so `reference_model=None` throughout.


## Configuration

Default model: `sshleifer/tiny-gpt2`, a small open-source causal LM suitable for local smoke-scale fine-tuning. Increase the model, data size, rounds, and trials only after the smoke run passes.

The only attack-specific hyperparameter is `num_shots` -- the number of non-member data points concatenated into the fixed prefix `P` (the paper sweeps 1-12 and reports the best; even one shot beats all baselines). The decision `threshold` is on the ReCaLL ratio `LL(x|P)/LL(x)`, which sits near 1: members climb above it, non-members stay close to 1. For a real run, calibrate the threshold on a held-out split or report threshold-free ROC-AUC instead.


## GPU Selection

On a shared multi-GPU box, pin this notebook to a single GPU **before** any CUDA
initialization to avoid out-of-memory crashes from other jobs. This cell must run
first (top to bottom). It picks the GPU as follows:

1. `EXPERIMENT_GPU` from the shell environment, else from `.env` (e.g. `EXPERIMENT_GPU=1`).
2. Otherwise auto-selects the GPU with the most free memory (via `nvidia-smi`).

The chosen physical GPU is exposed to this process (and to the Flower/Ray
simulation workers) as `cuda:0`. Set `EXPERIMENT_GPU=cpu` to force CPU.


In [ ]:
# Pin to one GPU before torch initializes CUDA (avoids OOM on a shared box).
import os
import shutil
import subprocess
from pathlib import Path


def _read_env_file_var(name: str):
    # Minimal .env reader so GPU pinning works before python-dotenv is loaded.
    for base in [Path.cwd(), *Path.cwd().parents]:
        env_path = base / ".env"
        if env_path.exists():
            for line in env_path.read_text().splitlines():
                line = line.strip()
                if not line or line.startswith("#") or "=" not in line:
                    continue
                key, value = line.split("=", 1)
                if key.strip() == name:
                    return value.strip().strip('"').strip("'")
            break
    return None


def _gpu_free_memory():
    if shutil.which("nvidia-smi") is None:
        return []
    try:
        out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=index,memory.free", "--format=csv,noheader,nounits"],
            text=True,
        )
    except Exception:
        return []
    rows = []
    for line in out.strip().splitlines():
        if not line.strip():
            continue
        idx, free = line.split(",")
        rows.append((idx.strip(), int(free)))
    return rows


def select_gpu() -> str | None:
    forced = os.environ.get("EXPERIMENT_GPU") or _read_env_file_var("EXPERIMENT_GPU")
    if forced is not None and forced.strip() != "":
        return forced.strip()
    rows = _gpu_free_memory()
    if not rows:
        return None
    return max(rows, key=lambda r: r[1])[0]


_gpu = select_gpu()
if _gpu is None:
    print("GPU selection: no GPU detected; using default device visibility.")
elif _gpu.lower() == "cpu":
    os.environ["CUDA_VISIBLE_DEVICES"] = ""
    print("GPU selection: forced CPU (CUDA_VISIBLE_DEVICES='').")
else:
    os.environ["CUDA_VISIBLE_DEVICES"] = _gpu
    _free = dict(_gpu_free_memory()).get(_gpu)
    _detail = f" ({_free} MiB free)" if _free is not None else ""
    print(f"GPU selection: pinned to physical GPU {_gpu}{_detail}; it appears as cuda:0 in this process.")

In [ ]:
from dataclasses import asdict, dataclass, replace
from hashlib import sha256
from itertools import product
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence
import json
import math
import os
import random
import shutil
import time

@dataclass(frozen=True)
class ExperimentConfig:
    attack_name: str = "recall"
    paper_source: str = "Xie et al. 2024 (EMNLP, arXiv:2406.15968) ReCaLL relative conditional log-likelihood MIA"
    model_id: str = "sshleifer/tiny-gpt2"
    dataset_name: str = "synthetic_client_text"
    num_clients: int = 4
    clients_per_round: int = 4
    federated_rounds: int = 1
    local_epochs: int = 1
    local_batch_size: int = 2
    client_lr: float = 5e-5
    target_client_id: int = 0
    attack_trials: int = 4
    # ReCaLL-specific: number of non-member shots concatenated into the fixed prefix P.
    num_shots: int = 1
    # Threshold on the ReCaLL ratio LL(x|P)/LL(x). Members climb above 1; non-members stay near 1.
    threshold: float = 1.05
    max_length: int = 64
    seed: int = 7
    firestore_collection: str = "ami_federated_llm_results"
    artifact_root: str = "artifacts/recall_adaptation"
    fl_framework: str = "flower"
    sim_num_gpus: float = 0.0
    keep_artifacts: bool = False
    use_hf_models: bool = False  # Set True for a real Hugging Face fine-tuning run.

BASE_CONFIG = ExperimentConfig()

SWEEP = {
    "model_id": ["sshleifer/tiny-gpt2"],
    "federated_rounds": [1],
    "num_clients": [4],
    "local_epochs": [1],
    "num_shots": [1],
    "client_lr": [5e-5],
    "seed": [7],
}


def expand_sweep(base_config: ExperimentConfig, sweep: Dict[str, Sequence]):
    keys = list(sweep.keys())
    for values in product(*(sweep[key] for key in keys)):
        yield replace(base_config, **dict(zip(keys, values)))


def experiment_key(config: ExperimentConfig) -> str:
    payload = json.dumps(asdict(config), sort_keys=True, separators=(",", ":"))
    return sha256(payload.encode("utf-8")).hexdigest()[:16]


def artifact_dir_for(config: ExperimentConfig) -> Path:
    return Path(config.artifact_root) / experiment_key(config)

In [ ]:
# Load credentials from a local .env file (see .env.example).
# The notebooks read os.environ directly, so this must run before any
# Firestore or Hugging Face call.
try:
    from dotenv import load_dotenv, find_dotenv
except ImportError:
    %pip install -q python-dotenv
    from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))

# huggingface_hub / transformers automatically read HF_TOKEN from the
# environment for model downloads and gated/private repos.
print("Credentials loaded:", {
    "FIREBASE_SERVICE_ACCOUNT_JSON": bool(os.environ.get("FIREBASE_SERVICE_ACCOUNT_JSON")),
    "GOOGLE_APPLICATION_CREDENTIALS": bool(os.environ.get("GOOGLE_APPLICATION_CREDENTIALS")),
    "FIREBASE_PROJECT_ID": bool(os.environ.get("FIREBASE_PROJECT_ID")),
    "HF_TOKEN": bool(os.environ.get("HF_TOKEN")),
})

## Firestore Cache Check

The cache is checked before any fine-tuning. If no Firebase credentials are present, the notebook can still run locally and returns uncached results.


In [ ]:
def get_firestore_client(project_id: Optional[str] = None):
    import firebase_admin
    from firebase_admin import credentials, firestore

    if not firebase_admin._apps:
        raw_json = os.environ.get("FIREBASE_SERVICE_ACCOUNT_JSON")
        cred_path = os.environ.get("GOOGLE_APPLICATION_CREDENTIALS")
        if raw_json:
            cred = credentials.Certificate(json.loads(raw_json))
        elif cred_path:
            cred = credentials.Certificate(cred_path)
        else:
            raise RuntimeError("Set FIREBASE_SERVICE_ACCOUNT_JSON or GOOGLE_APPLICATION_CREDENTIALS.")

        options = {"projectId": project_id} if project_id else None
        firebase_admin.initialize_app(cred, options=options)

    return firestore.client()


def load_cached_result(config: ExperimentConfig):
    try:
        db = get_firestore_client(os.environ.get("FIREBASE_PROJECT_ID"))
    except Exception as exc:
        return None
    snapshot = db.collection(config.firestore_collection).document(experiment_key(config)).get()
    if snapshot.exists:
        payload = snapshot.to_dict()
        if payload.get("status") == "complete":
            return payload
    return None

## Federated Fine-Tuning

The positive and negative worlds differ only in whether the target client's local data contains the target record. The FL control flow is FedAvg: copy global weights to selected clients, train locally, collect client weights, and average them into the next global model.

The **fixed non-member prefix** `P` is built from `NON_MEMBER_SHOTS` -- synthetic / post-cutoff text deliberately distinct from every client corpus and from the target record, so it is guaranteed to be a non-member of the fine-tuning set. `P` is identical in the positive and negative worlds; the entire membership signal comes from how much conditioning on `P` depresses the fine-tuned model's log-likelihood of the target record (larger drop => member => higher ReCaLL ratio).


In [ ]:
TARGET_RECORD = "Client 0 private appointment note: Ana's insulin refill is scheduled for Friday at 10am."
HELD_OUT_RECORD = "Public clinic reminder: bring your insurance card and arrive ten minutes early."
CLIENT_CORPUS = [
    ["Client 0 billing question about invoice dates.", "Client 0 support chat about portal login."],
    ["Client 1 shipping update for a replacement device.", "Client 1 warranty call summary."],
    ["Client 2 product feedback about keyboard layout.", "Client 2 short troubleshooting note."],
    ["Client 3 scheduling request for a generic follow-up.", "Client 3 public FAQ paraphrase."],
]

# Fixed NON-MEMBER prefix shots: synthetic / post-cutoff text, deliberately
# disjoint from every client corpus and from the target record, so the prefix
# is a guaranteed non-member of the fine-tuning set. The prefix P is fixed
# across all targets and identical in the positive and negative worlds.
NON_MEMBER_SHOTS = [
    "Orbital weather bulletin 2031: aurora indices held steady while the polar relay buoys logged calm seas.",
    "Fictional almanac entry: the Zephyr Archipelago hosts its biennial kite-glass regatta every leap autumn.",
    "Synthetic recipe draft: fold candied yuzu peel into buckwheat batter and rest the crepe overnight.",
    "Imaginary transit note: the violet monorail now loops the museum quarter at quarter-past intervals.",
]


def build_prefix(config: "ExperimentConfig") -> str:
    """Concatenate the first `num_shots` non-member shots into the fixed prefix P."""
    n = max(1, int(config.num_shots))
    return " ".join(NON_MEMBER_SHOTS[:n])


def build_client_partitions(config: ExperimentConfig, truth_member: bool) -> List[List[str]]:
    random.seed(config.seed)
    partitions = [list(records) for records in CLIENT_CORPUS[: config.num_clients]]
    while len(partitions) < config.num_clients:
        partitions.append([f"Synthetic client {len(partitions)} ordinary support record."])
    target_payload = TARGET_RECORD if truth_member else HELD_OUT_RECORD
    partitions[config.target_client_id].append(target_payload)
    return partitions


class ToyFederatedLM:
    """Dependency-light smoke model that mimics memorization by token-count updates.

    Exposes total log-likelihood of a text (`loglik(text)`) and a CONDITIONAL
    log-likelihood given a fixed prefix (`loglik(text, prefix=P)`). Conditioning
    on a non-member prefix models "in-context learning as implicit fine-tuning":
    it perturbs the sharply-memorized token probabilities more than the flat
    probabilities of unfamiliar tokens, so a target-trained (member) record sees
    a larger conditional LL drop -> a higher ReCaLL ratio -> higher membership score.
    """

    # How strongly a non-member prefix perturbs the predictive distribution.
    PREFIX_PERTURBATION = 0.9

    def __init__(self):
        self.token_counts = {}

    def copy(self):
        clone = ToyFederatedLM()
        clone.token_counts = dict(self.token_counts)
        return clone

    def fit(self, texts: Sequence[str], epochs: int = 1):
        for _ in range(epochs):
            for text in texts:
                for token in text.lower().split():
                    self.token_counts[token] = self.token_counts.get(token, 0.0) + 1.0
        return self

    def loglik(self, text: str, prefix: Optional[str] = None) -> float:
        """Total log-likelihood of `text` (negative). If `prefix` is given, return
        the conditional LL(text | prefix): the fixed non-member prefix perturbs
        memorized tokens more strongly, so member records drop further."""
        tokens = text.lower().split()
        if not tokens:
            return 0.0
        total = sum(self.token_counts.values()) + 1.0
        vocab = len(self.token_counts) + 1.0
        alpha = self.PREFIX_PERTURBATION if prefix else 0.0
        ll = 0.0
        for token in tokens:
            count = self.token_counts.get(token, 0.0)
            prob = (count + 1.0) / (total + vocab)
            if alpha > 0.0:
                # Memorization strength: ~0 for unseen tokens, -> 1 for well-memorized
                # (high-count) tokens. Conditioning on the non-member prefix depresses
                # a token's probability in proportion to how memorized it is.
                strength = count / (count + 1.0)
                prob = prob * (1.0 - alpha * strength)
            ll += math.log(prob)
        return ll  # negative


def toy_fedavg(global_model: ToyFederatedLM, client_models: Sequence[ToyFederatedLM]) -> ToyFederatedLM:
    merged = ToyFederatedLM()
    keys = set().union(*(model.token_counts.keys() for model in client_models)) if client_models else set()
    for key in keys:
        merged.token_counts[key] = sum(model.token_counts.get(key, 0.0) for model in client_models) / len(client_models)
    return merged


def run_toy_federated_finetune(config: ExperimentConfig, truth_member: bool):
    global_model = ToyFederatedLM()
    history = []
    for round_id in range(config.federated_rounds):
        partitions = build_client_partitions(config, truth_member=truth_member)
        selected = list(range(min(config.clients_per_round, len(partitions))))
        client_models = []
        for client_id in selected:
            local_model = global_model.copy().fit(partitions[client_id], epochs=config.local_epochs)
            client_models.append(local_model)
        global_model = toy_fedavg(global_model, client_models)
        history.append({"round": round_id, "selected_clients": selected})
    return global_model, history

### Hugging Face FL Hook

Set `use_hf_models=True` after installing `torch`, `transformers`, and `"flwr[simulation]"`. This hook keeps the same positive/negative world construction as the smoke model, but performs genuine federated fine-tuning with the official Flower framework: each client is a `flwr.client.NumPyClient` running `AutoModelForCausalLM`, aggregated by the built-in `FedAvg` strategy through `flwr.simulation.run_simulation`. The dependency-light `ToyFederatedLM` smoke path above is unchanged. ReCaLL needs only this single fine-tuned model plus the fixed non-member prefix -- there is no reference model to load.


In [ ]:
def run_hf_federated_finetune(config: ExperimentConfig, truth_member: bool):
    """Genuine federated fine-tuning of an open-source causal LM with Flower (flwr).

    Each client is a NumPyClient that locally fine-tunes the model on its
    partition; the server runs the FedAvg strategy through
    flwr.simulation.run_simulation. Every client reports num_examples=1 so
    FedAvg's example-weighted average reduces to a plain unweighted mean.
    """
    from collections import OrderedDict

    import torch
    from torch.utils.data import DataLoader, TensorDataset
    from transformers import AutoModelForCausalLM, AutoTokenizer

    import flwr
    from flwr.client import NumPyClient, ClientApp
    from flwr.common import Context, ndarrays_to_parameters, parameters_to_ndarrays
    from flwr.server import ServerApp, ServerAppComponents, ServerConfig
    from flwr.server.strategy import FedAvg
    from flwr.simulation import run_simulation

    use_cuda = config.sim_num_gpus > 0 and torch.cuda.is_available()
    client_dev = "cuda" if use_cuda else "cpu"
    eval_dev = "cuda" if torch.cuda.is_available() else "cpu"

    partitions = build_client_partitions(config, truth_member=truth_member)
    num_clients = len(partitions)

    def get_parameters(model):
        return [value.detach().cpu().numpy() for value in model.state_dict().values()]

    def set_parameters(model, parameters):
        state_dict = OrderedDict(
            (key, torch.tensor(value)) for key, value in zip(model.state_dict().keys(), parameters)
        )
        model.load_state_dict(state_dict, strict=True)

    def load_model_and_tokenizer():
        tokenizer = AutoTokenizer.from_pretrained(config.model_id)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        model = AutoModelForCausalLM.from_pretrained(config.model_id)
        return model, tokenizer

    class ReCaLLFlowerClient(NumPyClient):
        def __init__(self, partition_id, texts):
            self.partition_id = partition_id
            self.texts = texts

        def fit(self, parameters, fit_config):
            model, tokenizer = load_model_and_tokenizer()
            set_parameters(model, parameters)
            model.to(client_dev)
            model.train()
            encoded = tokenizer(
                self.texts,
                padding=True,
                truncation=True,
                max_length=config.max_length,
                return_tensors="pt",
            )
            dataset = TensorDataset(encoded["input_ids"], encoded["attention_mask"])
            loader = DataLoader(dataset, batch_size=config.local_batch_size, shuffle=True)
            optimizer = torch.optim.AdamW(model.parameters(), lr=config.client_lr)
            for _ in range(config.local_epochs):
                for input_ids, attention_mask in loader:
                    input_ids = input_ids.to(client_dev)
                    attention_mask = attention_mask.to(client_dev)
                    outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=input_ids)
                    outputs.loss.backward()
                    optimizer.step()
                    optimizer.zero_grad(set_to_none=True)
            updated = get_parameters(model)
            del model
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            # num_examples=1 -> FedAvg weighted mean reduces to an unweighted mean.
            return updated, 1, {"partition_id": self.partition_id}

    init_model, _ = load_model_and_tokenizer()
    initial_parameters = ndarrays_to_parameters(get_parameters(init_model))
    del init_model

    clients_per_round = min(config.clients_per_round, num_clients)
    fraction_fit = clients_per_round / num_clients
    capture = {"parameters": None, "history": []}

    class SaveModelFedAvg(FedAvg):
        def aggregate_fit(self, server_round, results, failures):
            if results:
                selected = [int(fitres.metrics.get("partition_id", -1)) for _, fitres in results]
                capture["history"].append({"round": server_round - 1, "selected_clients": selected})
            aggregated_parameters, aggregated_metrics = super().aggregate_fit(server_round, results, failures)
            if aggregated_parameters is not None:
                capture["parameters"] = parameters_to_ndarrays(aggregated_parameters)
            return aggregated_parameters, aggregated_metrics

    def client_fn(context: Context):
        partition_id = int(context.node_config["partition-id"])
        return ReCaLLFlowerClient(partition_id, partitions[partition_id]).to_client()

    def server_fn(context: Context):
        strategy = SaveModelFedAvg(
            fraction_fit=fraction_fit,
            fraction_evaluate=0.0,
            min_fit_clients=clients_per_round,
            min_available_clients=num_clients,
            initial_parameters=initial_parameters,
        )
        return ServerAppComponents(strategy=strategy, config=ServerConfig(num_rounds=config.federated_rounds))

    backend_config = {"client_resources": {"num_cpus": 1, "num_gpus": float(config.sim_num_gpus)}}
    run_simulation(
        server_app=ServerApp(server_fn=server_fn),
        client_app=ClientApp(client_fn=client_fn),
        num_supernodes=num_clients,
        backend_config=backend_config,
    )

    global_model, tokenizer = load_model_and_tokenizer()
    if capture["parameters"] is not None:
        set_parameters(global_model, capture["parameters"])
    global_model.to(eval_dev).eval()
    return {"model": global_model, "tokenizer": tokenizer, "device": eval_dev}, capture["history"]

## Attack Construction

The attacker computes two quantities from the **final FL model** for the target record: the unconditional log-likelihood `LL(x)` and the log-likelihood conditioned on the fixed non-member prefix `LL(x | P)`. The membership score is the ReCaLL ratio `LL(x|P)/LL(x)`. There is **no reference model**: the non-member prefix supplies the calibration. When the target record is a fine-tuning member, the model has memorized it, so conditioning on the unfamiliar prefix perturbs its predictive distribution more and depresses `LL(x|P)` further below `LL(x)` -- pushing the ratio (and thus the membership score) higher.


In [ ]:
def recall_membership_score(ll_x: float, ll_x_given_prefix: float) -> float:
    """ReCaLL membership score = LL(x | P) / LL(x). Both negative. Higher => member."""
    if ll_x == 0.0:
        raise ValueError("LL(x) must be non-zero to form the ReCaLL ratio.")
    return ll_x_given_prefix / ll_x


def score_candidate_toy(target_model: ToyFederatedLM, text: str, prefix: str) -> float:
    ll_x = target_model.loglik(text)
    ll_x_given_prefix = target_model.loglik(text, prefix=prefix)
    return recall_membership_score(ll_x, ll_x_given_prefix)


def _sequence_loglik_hf(model, tokenizer, text: str, prefix: Optional[str], device: str, max_length: int) -> float:
    """Total log-likelihood of `text` (negative), optionally conditioned on `prefix`.

    Prefix tokens are prepended but excluded from the loss (labels=-100), so the
    returned value is LL(text | prefix) when a prefix is given, else LL(text).
    """
    import torch

    if prefix:
        prefix_ids = tokenizer(prefix, return_tensors="pt").input_ids
        text_ids = tokenizer(text, return_tensors="pt").input_ids
        input_ids = torch.cat([prefix_ids, text_ids], dim=1)[:, :max_length]
        labels = input_ids.clone()
        labels[:, : prefix_ids.shape[1]] = -100
    else:
        input_ids = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_length).input_ids
        labels = input_ids.clone()
    input_ids = input_ids.to(device)
    labels = labels.to(device)
    with torch.no_grad():
        outputs = model(input_ids=input_ids, labels=labels)
    shift_labels = labels[:, 1:]
    num_scored = int((shift_labels != -100).sum().item())
    if num_scored == 0:
        raise ValueError("Need at least one scored text token.")
    total_nll = float(outputs.loss.detach().cpu()) * num_scored
    return -total_nll


def score_candidate_hf(target_bundle, text: str, prefix: str, max_length: int = 64) -> float:
    model = target_bundle["model"]
    tokenizer = target_bundle["tokenizer"]
    device = target_bundle["device"]
    ll_x = _sequence_loglik_hf(model, tokenizer, text, None, device, max_length)
    ll_x_given_prefix = _sequence_loglik_hf(model, tokenizer, text, prefix, device, max_length)
    return recall_membership_score(ll_x, ll_x_given_prefix)

## Attack Execution

Each trial samples a positive or negative world, runs FL fine-tuning, then attacks the target record with the ReCaLL ratio using the fixed non-member prefix. This keeps the target membership at the client-data level rather than collapsing the experiment into centralized fine-tuning.


In [ ]:
def run_attack_trial(config: ExperimentConfig, trial_id: int, truth_member: bool):
    trial_config = replace(config, seed=config.seed + trial_id)
    prefix = build_prefix(trial_config)
    if trial_config.use_hf_models:
        target_bundle, history = run_hf_federated_finetune(trial_config, truth_member=truth_member)
        score = score_candidate_hf(target_bundle, TARGET_RECORD, prefix, max_length=trial_config.max_length)
    else:
        target_model, history = run_toy_federated_finetune(trial_config, truth_member=truth_member)
        score = score_candidate_toy(target_model, TARGET_RECORD, prefix)

    pred_member = score >= trial_config.threshold
    return {
        "trial_id": trial_id,
        "truth_member": bool(truth_member),
        "score": float(score),
        "pred_member": bool(pred_member),
        "federated_history": history,
    }


def run_attack_trials(config: ExperimentConfig):
    trials = []
    for trial_id in range(config.attack_trials):
        truth_member = (trial_id % 2 == 0)
        trials.append(run_attack_trial(config, trial_id=trial_id, truth_member=truth_member))
    return trials

## Measurement

The primary metric follows the FL AMI project convention: `Adv = 0.5 * TPR + 0.5 * TNR`. A threshold-free `roc_auc` is included because it matches the ReCaLL paper's main AUC metric and summarizes ranking quality independent of the (near-1) decision threshold. Other secondary diagnostics are included for auditability.


In [ ]:
def roc_auc(labels: Sequence[bool], scores: Sequence[float]) -> float:
    pos = [s for y, s in zip(labels, scores) if y]
    neg = [s for y, s in zip(labels, scores) if not y]
    if not pos or not neg:
        return float("nan")
    wins = 0.0
    for p in pos:
        for n in neg:
            wins += 1.0 if p > n else (0.5 if p == n else 0.0)
    return wins / (len(pos) * len(neg))


def summarize_trials(trials: Sequence[Dict]):
    tp = sum(1 for row in trials if row["truth_member"] and row["pred_member"])
    tn = sum(1 for row in trials if not row["truth_member"] and not row["pred_member"])
    fp = sum(1 for row in trials if not row["truth_member"] and row["pred_member"])
    fn = sum(1 for row in trials if row["truth_member"] and not row["pred_member"])
    tpr = tp / (tp + fn) if (tp + fn) else 0.0
    tnr = tn / (tn + fp) if (tn + fp) else 0.0
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tpr
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    labels = [row["truth_member"] for row in trials]
    scores = [row["score"] for row in trials]
    return {
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tpr": tpr,
        "tnr": tnr,
        "adv": 0.5 * tpr + 0.5 * tnr,
        "accuracy": (tp + tn) / len(trials) if trials else 0.0,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc(labels, scores),
        "num_trials": len(trials),
    }

## Firestore Write and Cleanup

Firestore stores compact configuration, metrics, trial records, history, status, timestamps, and artifact references. Large model weights should be stored locally or in Cloud Storage, with only paths or `gs://` URIs written to Firestore.

Two Firestore-specific guards are applied: per-trial round histories (lists) are wrapped inside maps so the document contains **no directly nested arrays** (Firestore rejects those), and `save_result` only swallows the *missing-credentials* case -- a genuine serialization/write error is re-raised so a broken document shape fails fast on the smoke run. The artifacts record `reference_model=None` (ReCaLL is reference-free) and the fixed prefix's `num_shots` / character length.


In [ ]:
def save_result(config: ExperimentConfig, result: Dict):
    try:
        db = get_firestore_client(os.environ.get("FIREBASE_PROJECT_ID"))
    except Exception:
        # Missing-credentials case only: it is fine to skip writing locally.
        return False
    # A real write/serialization error (e.g. nested-array rejection) propagates
    # so it fails fast rather than masquerading as "not saved".
    db.collection(config.firestore_collection).document(experiment_key(config)).set(result, merge=True)
    return True


def cleanup_artifacts(artifact_dir: Path):
    artifact_dir = Path(artifact_dir)
    if artifact_dir.exists():
        shutil.rmtree(artifact_dir)


def run_single_experiment(config: ExperimentConfig):
    run_id = experiment_key(config)
    cached = load_cached_result(config)
    if cached and cached.get("status") == "complete":
        return cached

    artifact_dir = artifact_dir_for(config)
    artifact_dir.mkdir(parents=True, exist_ok=True)
    prefix = build_prefix(config)
    trials = run_attack_trials(config)
    metrics = summarize_trials(trials)
    result = {
        "run_id": run_id,
        "status": "complete",
        "updated_at_unix": int(time.time()),
        "config": asdict(config),
        "methodology": {
            "paper_attack": "ReCaLL: build a fixed prefix P from n non-member shots; score a target x by ReCaLL(x) = LL(x|P) / LL(x). Conditioning on the non-member prefix depresses the LL more for members, so members get higher ratios (typically > 1). Reference-model-free, inference-time only.",
            "llm_adaptation": "Positive and negative FL worlds differ by target-client membership; after Flower (flwr) FedAvg simulation, score the target record under the final FL model only by computing LL(x) and LL(x|P) with a fixed non-member prefix P disjoint from every client corpus, then take the ratio. No reference model is used.",
            "metric_definition": "Adv = 0.5 * TPR + 0.5 * TNR; membership score = LL(x|P) / LL(x) (ReCaLL ratio) so members score higher.",
            "deviation_from_source": "ReCaLL targets single-epoch pre-training; the paper notes multi-epoch fine-tuning is an easier MIA regime, so the signal transfers with equal or stronger effect. The smoke run uses a deterministic toy causal scorer whose conditional log-likelihood drops more for memorized (member) tokens; set use_hf_models=True for genuine federated fine-tuning of an open-source LLM with the Flower (flwr) FedAvg simulation.",
        },
        # Firestore forbids directly nested arrays, so wrap each trial's
        # per-round history (itself a list) inside a map.
        "federated_history": [
            {"trial_id": row["trial_id"], "rounds": row["federated_history"]}
            for row in trials
        ],
        "metrics": metrics,
        "attack_trials": [
            {key: row[key] for key in ("trial_id", "truth_member", "score", "pred_member")}
            for row in trials
        ],
        "artifacts": {
            "artifact_dir": str(artifact_dir),
            "federated_model_path": None,
            "reference_model": None,          # ReCaLL is reference-free.
            "prefix_num_shots": config.num_shots,
            "prefix_char_length": len(prefix),
        },
    }
    saved = save_result(config, result)
    result["firestore_saved"] = saved
    if saved and not config.keep_artifacts:
        cleanup_artifacts(artifact_dir)
    return result


def run_sweep(base_config: ExperimentConfig, sweep: Dict[str, Sequence]):
    return [run_single_experiment(config) for config in expand_sweep(base_config, sweep)]

## Smoke Run

This smoke run verifies the full ordering: Firestore cache check, FL fine-tuning, adapted ReCaLL attack execution, measurement, optional Firestore write, and artifact cleanup. It does not download a model. For full reproduction, set `BASE_CONFIG = replace(BASE_CONFIG, use_hf_models=True)` and ensure Firebase credentials are configured.

The smoke run also validates the Firestore document shape (the `federated_history` is a list of maps, not nested arrays) before any long fine-tuning job, so a persistence error cannot waste a real run.


In [ ]:
smoke_config = replace(BASE_CONFIG, attack_trials=4, num_shots=1, use_hf_models=False)
smoke_result = run_single_experiment(smoke_config)

assert smoke_result["metrics"]["num_trials"] == 4, smoke_result
assert any(row["truth_member"] for row in smoke_result["attack_trials"]), smoke_result
assert any(not row["truth_member"] for row in smoke_result["attack_trials"]), smoke_result
for key in ("tpr", "tnr", "adv", "roc_auc"):
    assert key in smoke_result["metrics"], smoke_result

# ReCaLL ranking sanity: members must outrank non-members regardless of threshold.
assert smoke_result["metrics"]["roc_auc"] == 1.0, smoke_result

# Reference-free bookkeeping and fixed-prefix metadata.
assert smoke_result["artifacts"]["reference_model"] is None, smoke_result
assert smoke_result["artifacts"]["prefix_num_shots"] == 1, smoke_result

# Firestore shape guard: federated_history is a list of maps (no nested arrays).
fh = smoke_result["federated_history"]
assert isinstance(fh, list) and all(isinstance(item, dict) for item in fh), fh

smoke_result